In [1]:
# Imports + setup
import random, pandas as pd, numpy as np, torch, textattack
from pathlib import Path  # Import Path for filesystem paths
from transformers import AutoTokenizer, BertForSequenceClassification, TrainingArguments, Trainer  # Import Transformers components
from textattack.datasets import Dataset as TA_Dataset  # Import TextAttack dataset wrapper
from textattack.models.wrappers import HuggingFaceModelWrapper  # Import HF wrapper for TextAttack
from textattack.attack_recipes import TextFoolerJin2019  # Import TextFooler recipe
from textattack.attack_recipes import BAEGarg2019 # Import BAE recipe
from collections import Counter  # Import Counter for label distribution reporting
from IPython.display import display  # Import display for displaying tables
from sklearn.metrics import accuracy_score, precision_recall_fscore_support # Import metrics to calculate f1, precision, recall, accuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import json

SEED = 2025  # Set global random seed
ANALYSIS_DIR = Path("../analysis/text_attack")  # Set analysis directory path
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)  # Create analysis directory if missing

random.seed(SEED)              # Set Python random seed
np.random.seed(SEED)           # Set Numpy random seed
torch.manual_seed(SEED)        # Set Torch CPU seed
if torch.cuda.is_available():  # Check CUDA availability
    torch.cuda.manual_seed_all(SEED)  # Set CUDA seed for all devices


2025-11-06 00:21:15.363448: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-06 00:21:15.489548: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-06 00:21:16.721024: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/jieba/_compat.py:18:

In [2]:
# Set the best_hyperparameters to default values initially
best_hyperparameters = {                # Define default hyperparameters
    "learning_rate": 2e-5,              # Set default learning rate
    "num_train_epochs": 3,              # Set default epochs
    "per_device_train_batch_size": 16,  # Set default batch size
    "weight_decay": 0.01,               # Set default weight decay
    "dropout": 0.1,                     # Set default dropout
}

optuna_database_path = Path.cwd() / "tuning_checkpoints" / "bert_tuning_full.db"  # Set Optuna DB path
optuna_study_name = "bert_tuning_full"  # Set Optuna study name

try:  # Try loading Optuna study
    import optuna  # Import Optuna
    loaded_study = optuna.load_study(  # Load study
        study_name=optuna_study_name,
        storage=f"sqlite:///{optuna_database_path}",
    )
    best_hyperparameters.update(loaded_study.best_trial.params)  # Update defaults with best trial
    print("Loaded best hyperparameters from bert_tuning_full.db")  
except Exception: 
    print("Optuna study not found, using default hyperparameters.")  

# Normalize and cast parameter types
best_hyperparameters["num_train_epochs"] = int(best_hyperparameters.get("num_train_epochs", 3))                         # Cast epochs to int
best_hyperparameters["per_device_train_batch_size"] = int(best_hyperparameters.get("per_device_train_batch_size", 16))  # Cast batch size to int
best_hyperparameters["learning_rate"] = float(best_hyperparameters.get("learning_rate", 2e-5))                          # Cast learning rate to float
best_hyperparameters["weight_decay"] = float(best_hyperparameters.get("weight_decay", 0.01))                            # Cast weight decay to float
best_hyperparameters["dropout"] = float(best_hyperparameters.get("dropout", 0.1))                                       # Cast dropout to float

# Print final hyperparameters for reference
print("\nFinal hyperparameters in use:")
for key, value in best_hyperparameters.items():
    print(f"  {key}: {value}")


Loaded best hyperparameters from bert_tuning_full.db

Final hyperparameters in use:
  learning_rate: 1.826222907426669e-05
  num_train_epochs: 5
  per_device_train_batch_size: 16
  weight_decay: 0.10631544814686143
  dropout: 0.3098949027144738


In [3]:
# Load tokenizer + model checkpoint
model_checkpoint_path = Path("../models") / "saved_model"  # Set model checkpoint path
model_name_or_path = str(model_checkpoint_path) if model_checkpoint_path.exists() else "bert-base-uncased"  # Choose model source

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)  # Load tokenizer
model = BertForSequenceClassification.from_pretrained(  # Load classification model
    model_name_or_path,   # Model using
    num_labels=2,         # Indicate possible results
    hidden_dropout_prob=best_hyperparameters["dropout"],
    attention_probs_dropout_prob=best_hyperparameters["dropout"],
)  
if torch.cuda.is_available():
    model = model.to("cuda")  # Move model to GPU (Recommended because it is faster)

In [4]:
# Load precomputed splits
splits_dir_path = Path("../data") / "splits"    # Set splits directory path
train_csv_path = splits_dir_path / "train.csv"  # Set train CSV path
val_csv_path = splits_dir_path / "val.csv"      # Set validation CSV path
test_csv_path = splits_dir_path / "test.csv"    # Set test CSV path

train_df = pd.read_csv(train_csv_path)   # Load train split
val_df = pd.read_csv(val_csv_path)       # Load validation split
test_df = pd.read_csv(test_csv_path)     # Load test split

print("Loaded precomputed splits from data/splits/.")

train_texts = train_df["text"].astype(str).tolist()       # Extract train texts
train_labels = train_df["label"].astype(int).tolist()     # Extract train labels
val_texts = val_df["text"].astype(str).tolist()        # Extract val texts
val_labels = val_df["label"].astype(int).tolist()      # Extract val labels
test_texts = test_df["text"].astype(str).tolist()         # Extract test texts
test_labels = test_df["label"].astype(int).tolist()       # Extract test labels

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")  # Print split sizes


Loaded precomputed splits from data/splits/.
Train: 143810 | Val: 15979 | Test: 39948


In [5]:
# Build TextAttack dataset from correctly classified samples
candidate_size = 2000  # ***IMPORTANT*** Set number of candidate samples to check
batch_size = 32        # Set batch size for prediction
candidate_texts = train_texts[:candidate_size]     # Select candidate texts
candidate_labels = train_labels[:candidate_size]   # Select candidate labels
model_device = next(model.parameters()).device     # Infer model device

model.eval()  # Set model to eval mode

correct_texts = []    # Initialize list for correctly classified texts
correct_labels = []   # Initialize list for correctly classified labels

for start_idx in range(0, len(candidate_texts), batch_size):         # Iterate batched
    batch_texts = candidate_texts[start_idx:start_idx + batch_size]  # Slice batch texts
    encodings = tokenizer(  # Tokenize batch
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    encodings = {k: v.to(model_device) for k, v in encodings.items()}  # Move to device

    # Do prediction but don't train/learn
    with torch.no_grad():  # Disable gradients
        outputs = model(**encodings)  # Forward pass
        logits = outputs.logits       # Extract logits
        preds = logits.argmax(dim=1).cpu().numpy().tolist()  # Get predicted labels

    true_batch_labels = candidate_labels[start_idx:start_idx + batch_size]  # Slice true labels

    for text_value, true_label_value, pred_value in zip(batch_texts, true_batch_labels, preds):  # Compare predictions
        if int(true_label_value) == int(pred_value):  # Check correctness
            correct_texts.append(text_value)              # Store correct text
            correct_labels.append(int(true_label_value))  # Store correct label

print("Candidate slice distribution:", Counter(candidate_labels))     # Print original distribution
print("Correctly classified distribution:", Counter(correct_labels))  # Print filtered distribution

N_ATTACK = 500  # ***IMPORTANT*** Set total samples to attack
per_class = N_ATTACK // 2  # Set per-class target

correct_by_label = {0: [], 1: []}  # Initialize grouping dict
for text_value, lbl in zip(correct_texts, correct_labels):  # Group correct samples
    correct_by_label[int(lbl)].append((text_value, int(lbl)))  # Append tuple

avail_pos = len(correct_by_label[1])  # Count positives
avail_neg = len(correct_by_label[0])  # Count negatives
print("Available correctly classified per class - pos:", avail_pos, "neg:", avail_neg) 

sampled_pos = random.sample(correct_by_label[1], min(per_class, avail_pos))  # Sample positives
sampled_neg = random.sample(correct_by_label[0], min(per_class, avail_neg))  # Sample negatives

sampled_pairs = sampled_pos + sampled_neg  # Combine samples

# Separate the tuples
attack_texts_subset = [t for t, _ in sampled_pairs]   # Extract texts
attack_labels_subset = [l for _, l in sampled_pairs]  # Extract labels

textattack_examples = [(t, l) for t, l in zip(attack_texts_subset, attack_labels_subset)]  # Build TA examples
ta_dataset = TA_Dataset(textattack_examples)  # Wrap into TextAttack dataset

print("Prepared TextAttack dataset size:", len(textattack_examples))         # Print dataset size
print("Attack label counts:", Counter([l for _, l in textattack_examples]))  # Print label counts


Candidate slice distribution: Counter({0: 1088, 1: 912})
Correctly classified distribution: Counter({0: 1084, 1: 902})
Available correctly classified per class - pos: 902 neg: 1084
Prepared TextAttack dataset size: 500
Attack label counts: Counter({1: 250, 0: 250})


In [6]:
# Configure TextAttack recipes
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)  # Wrap model for TextAttack
available_recipes = []  # Initialize list of recipes
available_recipes.append(("TextFooler", TextFoolerJin2019.build(model_wrapper)))  # Add TextFooler recipe
available_recipes.append(("BAE", BAEGarg2019.build(model_wrapper)))               # Add BAE to list

print("Configured recipes:", [name for name, _ in available_recipes])  # Print configured recipes


textattack: Unknown if model of class <class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
textattack: Unknown if model of class <class 'transformers.mode

Configured recipes: ['TextFooler', 'BAE']


In [7]:
# Run TextAttack for each available recipe
def run_attack_recipe(recipe_name, recipe_object, ta_dataset):
    attack_args = textattack.AttackArgs(      # Configure attack arguments
        num_examples=len(ta_dataset),         # Use all samples in dataset
        log_to_csv=str(ANALYSIS_DIR / f"textattack_{recipe_name}.csv"),  # Set CSV log path
        checkpoint_interval=50,               # Set checkpoint interval
        parallel=False,                       # Disable parallel for notebook stability
        silent=True,                          # Suppress output
    )
    attacker = textattack.Attacker(  # Create attacker
        recipe_object,
        ta_dataset,
        attack_args=attack_args,
    )
    for _ in attacker.attack_dataset():  # Iterate over attacks to trigger logging
        pass  # Logging handled by TextAttack

for recipe_name_value, recipe_object_value in available_recipes:  # Run all recipes
    try:  # Handle individual recipe failure
        run_attack_recipe(recipe_name_value, recipe_object_value, ta_dataset)  # Run current recipe
    except Exception as attack_error:  # Print failure
        print(f"Recipe {recipe_name_value} failed to run:", attack_error) 


  0%|                                                                                           | 0/500 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1762358701.901352    2144 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3505 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9
[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   0%|                           | 1/500 [00:13<1:53:10, 13.61s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

substantial increase in length bigger dimension will also increase the endurance of your intercourse...get your love gun appreciated at its true value! [url] the potentialoffered not a rational view of the future but a biased one.acquisitions.


[Succeeded / Failed / Skipped / Total] 1 / 1 / 0 / 2:   0%|                           | 2/500 [00:27<1:52:41, 13.58s/it]

--------------------------------------------- Result 2 ---------------------------------------------
[[0 (100%)]] --> [[1 (74%)]]

[[re]]:[ilug] autorun [[cds]] apologies all.i [[have]] comitted a [[cardinal]] [[sin]] by not specifying that this was for a windows [[machine]].----- [[original]] [[message]] ----- from:"kenn humborg" to:"[[david]] crozier" [[cc]]:[[sent]]:[[friday]], [[august]] 16, 2002 1:51 [[pm]] subject:[[re]]:[ilug] autorun cds > > [[cheers]] all for your [[words]] of [[wisdom]].> > > > i [[came]] across this which [[has]] [[worked]] a [[treat]]:- > > > > [url] > > (this is all windows-related autorun [[stuff]]).> > [[so]] why [[did]] you [[waste]] the [[time]] of those who [[looked]] > up linux-related [[info]] for you? if you [[said]] that it > was for windows, you'd [[probably]] [[have]] [[gotten]] both > more [[relevant]] [[answers]] and [[flames]] for [[asking]] this on > a _linux_ mailing [[list]].> > [[sheesh]]...> > [[later]], > kenn > -- irish [[linux]] users

[Succeeded / Failed / Skipped / Total] 2 / 1 / 0 / 3:   1%|▏                          | 3/500 [00:30<1:23:33, 10.09s/it]

...

--------------------------------------------- Result 499 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

eol average deal count by trader and product as of 7 - 23 - 01 the following file contains a graphical view of the north american gas average deal count by trader and product for eol.this information is for comparative analysis only.do not update links when opening this file.if you have any questions regarding this breakout , please let me know.laura levy enrononline x 53551


[Succeeded / Failed / Skipped / Total] 73 / 427 / 0 / 500: 100%|████████████████████| 500/500 [2:23:28<00:00, 17.22s/it]

--------------------------------------------- Result 500 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

new dynegy logo i can't take credit for this, but am sharing with you all anyway....





[Succeeded / Failed / Skipped / Total] 73 / 427 / 0 / 500: 100%|████████████████████| 500/500 [2:23:28<00:00, 17.22s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 73     |
| Number of failed attacks:     | 427    |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 85.4%  |
| Attack success rate:          | 14.6%  |
| Average perturbed word %:     | 11.81% |
| Average num. words per input: | 229.12 |
| Avg num queries:              | 525.28 |
+-------------------------------+--------+


In [12]:
# Merge TextAttack CSV logs and split success/failed
merged_path = ANALYSIS_DIR / "textattack_merged.csv"                  # Set merged path
merged_success_path = ANALYSIS_DIR / "textattack_merged_success.csv"  # Set success path
merged_failed_path = ANALYSIS_DIR / "textattack_merged_failed.csv"    # Set failed path

dfs = []  # Initialize list for per-recipe DataFrames
for p in [ANALYSIS_DIR / "textattack_TextFooler.csv", ANALYSIS_DIR / "textattack_BAE.csv"]:  # Iterate expected files
    if p.exists():  # Check existence
        df = pd.read_csv(p)      # Load CSV
        df["__source"] = p.stem  # Add source tag
        dfs.append(df)           # Append to list
    else:  # Handle missing file
        print(f"WARNING: not found: {p}")  # Print warning

if not dfs:  # Check for empty load
    raise FileNotFoundError("No TextAttack CSV logs found to merge.")  # Raise error

df_attacks = pd.concat(dfs, ignore_index=True)  # Merge all logs
print(f"Merged rows: {len(df_attacks)}")  # Print merge size

orig_col = "original_text"         # Define original text column
pert_col = "perturbed_text"        # Define perturbed text column
result_col = "result_type"         # Define result type column
orig_out_col = "original_output"   # Define original output column
pert_out_col = "perturbed_output"  # Define perturbed output column

if result_col in df_attacks.columns:  # Derive success from result_type
    c = df_attacks[result_col].astype(str).str.strip().str.lower()  # Normalize result_type text
    is_success = c.eq("successful")    # Mark rows where result_type equals 'successful'
else:  # Fallback if column unexpectedly missing
    is_success = pd.Series([False] * len(df_attacks), index=df_attacks.index)  # Default to all failed

df_attacks["is_success"] = is_success.astype(bool)  # Store success flag

print(" total :", len(df_attacks))  # Print total rows
print(" success:", int(df_attacks["is_success"].sum()))     # Print success rows
print(" failed :", int((~df_attacks["is_success"]).sum()))  # Print failed rows

df_attacks.to_csv(merged_path, index=False, encoding="utf-8")  # Save merged CSV
df_attacks[df_attacks["is_success"]].to_csv(merged_success_path, index=False, encoding="utf-8")  # Save success CSV
df_attacks[~df_attacks["is_success"]].to_csv(merged_failed_path, index=False, encoding="utf-8")  # Save failed CSV

print(f"Saved merged to   : {merged_path}")              # Print path
print(f"Saved success rows to : {merged_success_path}")  # Print path
print(f"Saved failed rows to  : {merged_failed_path}")   # Print path

df_success = df_attacks[df_attacks["is_success"]].copy()  # Filter succeeded rows
df_failed = df_attacks[~df_attacks["is_success"]].copy()  # Filter failed rows

display_cols = [col for col in [orig_col, pert_col, orig_out_col, pert_out_col, result_col, "__source"] if col in df_attacks.columns]  # Select available columns

print("\nSAMPLE successful rows (first 5):")  # Print header
display(df_success[display_cols].head(5))     # Display first 5 success rows

print("\nSAMPLE failed rows (first 5):")      # Print header
display(df_failed[display_cols].head(5))      # Display first 5 failed rows


Merged rows: 1000
 total : 1000
 success: 443
 failed : 557
Saved merged to   : ../analysis/text_attack/textattack_merged.csv
Saved success rows to : ../analysis/text_attack/textattack_merged_success.csv
Saved failed rows to  : ../analysis/text_attack/textattack_merged_failed.csv

SAMPLE successful rows (first 5):


,original_text,perturbed_text,original_output,perturbed_output,result_type,__source
1,[[re]]:[ilug] autorun [[cds]] apologies all.i ...,[[rey]]:[ilug] autorun [[albums]] apologies al...,0,1,Successful,textattack_TextFooler
2,just [[take]] a look at this young ladys [[goo...,just [[toma]] a look at this young ladys [[exe...,1,0,Successful,textattack_TextFooler
4,[cc-devel] [ cctools-feature requests-1723155 ...,[cc-devel] [ cctools-feature requests-1723155 ...,0,1,Successful,textattack_TextFooler
5,isc - customer service [[survey]] isc - custom...,isc - customer service [[researches]] isc - cu...,0,1,Successful,textattack_TextFooler
6,"8 [[x]] longer than vlagra , and [[cheaper]] ,...","8 [[tenth]] longer than vlagra , and [[smaller...",1,0,Successful,textattack_TextFooler



SAMPLE failed rows (first 5):


,original_text,perturbed_text,original_output,perturbed_output,result_type,__source
0,substantial [[increase]] in [[length]] [[bigge...,substantial [[gains]] in [[lengths]] [[higher]...,1,1,Failed,textattack_TextFooler
3,top pills at [[low]] [[prices]] [[dear]] value...,top pills at [[mild]] [[expenditure]] [[cared]...,1,1,Failed,textattack_TextFooler
8,[[re]]:[[handy]] [[board]] [[troubles]]...that...,[[rey]]:[[valuable]] [[council]] [[commotion]]...,0,0,Failed,textattack_TextFooler
10,[[offer]] [[variety]] [[replica]] [[watches]] ...,[[delivering]] [[assorted]] [[replicate]] [[cl...,1,1,Failed,textattack_TextFooler
13,fw:[[transition]] [[services]] [[agreement]] g...,fw:[[transforming]] [[amenities]] [[convention...,0,0,Failed,textattack_TextFooler


In [13]:
# Build augmented train set from original + succeeded TA samples
success_csv_path = ANALYSIS_DIR / "textattack_merged_success.csv"  # Set success CSV path
merged_csv_path = ANALYSIS_DIR / "textattack_merged.csv"  # Set merged CSV path

augmented_train_texts = list(train_texts)    # Start from original train texts
augmented_train_labels = list(train_labels)  # Start from original train labels

print("Original training set size:", len(augmented_train_texts))  # Print initial size

adv_texts = []   # Initialize list for adversarial texts
adv_labels = []  # Initialize list for adversarial labels

if success_csv_path.exists():  # Prefer success-only file
    df_succ = pd.read_csv(success_csv_path)  # Load file
    if "is_success" in df_succ.columns:
        df_succ = df_succ[df_succ["is_success"] == True]   # Filter succeeded rows
    if "perturbed_text" in df_succ.columns and df_succ["perturbed_text"].astype(str).str.len().gt(0).any():
        adv_texts = df_succ["perturbed_text"].astype(str).tolist()  # Extract perturbed texts
        if "ground_truth_output" in df_succ.columns:
            adv_labels = df_succ["ground_truth_output"].astype(int).tolist()  # Use ground truth labels
        else:
            adv_labels = df_succ["original_output"].astype(int).tolist()  # Fallback to original output
        print(f"Loaded {len(adv_texts)} adversarial samples from success CSV.")  # Print count
else:
    print("Cannot find the Successful Attacked Cases.") 

if len(adv_texts) == len(adv_labels) and len(adv_texts) > 0:  # Verify alignment
    augmented_train_texts.extend(adv_texts)                 # Append texts
    augmented_train_labels.extend(adv_labels)               # Append labels
    print("Appended adversarial samples to training set.")  # Print append notice
elif len(adv_texts) != len(adv_labels):
    print("WARNING: texts and labels length mismatch, skipping append.")  # Print mismatch
else:
    print("No adversarial samples added.")  # Print nothing added

combined = list(zip(augmented_train_texts, augmented_train_labels))  # Pair texts and labels
random.shuffle(combined)  # Shuffle combined list
augmented_train_texts, augmented_train_labels = zip(*combined)  # Unzip back
augmented_train_texts = list(augmented_train_texts)             # Convert to list
augmented_train_labels = list(augmented_train_labels)           # Convert to list

print("Final augmented training set size:", len(augmented_train_texts))  # Print final train size
print("Validation set size:", len(val_texts))  # Print val size


Original training set size: 143810
Loaded 443 adversarial samples from success CSV.
Appended adversarial samples to training set.
Final augmented training set size: 144253
Validation set size: 15979


In [14]:
# Tokenize augmented train + validation
aug_train_encodings = tokenizer(  # Tokenize augmented training texts
    augmented_train_texts,
    truncation=True,
    padding=True,
    max_length=256,
)

val_encodings = tokenizer(        # Tokenize validation texts
    val_texts,
    truncation=True,
    padding=True,
    max_length=256,
)

print("Tokenized augmented train samples:", len(aug_train_encodings["input_ids"]))  # Print train token count
print("Tokenized validation samples:", len(val_encodings["input_ids"]))             # Print val token count


Tokenized augmented train samples: 144253
Tokenized validation samples: 15979


In [15]:
# Define dataset objects
import torch  # Torch already imported, kept for clarity in this cell

class EmailDataset(torch.utils.data.Dataset):  # Define custom dataset
    def __init__(self, encodings, labels):  # Init dataset
        self.encodings = encodings          # Store encodings
        self.labels = list(labels)          # Store labels

    def __len__(self):  # Define length
        return len(self.labels)             # Return number of samples

    def __getitem__(self, idx):  # Define item getter
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}     # Build features dict
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)  # Add label tensor
        return item  # Return item

train_dataset_aug = EmailDataset(aug_train_encodings, augmented_train_labels)  # Build train dataset
val_dataset = EmailDataset(val_encodings, val_labels)                          # Build validation dataset

print("Train dataset length:", len(train_dataset_aug))  # Print train dataset length
print("Val dataset length:", len(val_dataset))          # Print val dataset length


Train dataset length: 144253
Val dataset length: 15979


In [16]:
# Custom evaluation metrics function
def compute_metrics(eval_pred):
    # Support both tuple and EvalPrediction objects
    if hasattr(eval_pred, "predictions"):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        logits, labels = eval_pred

    # Convert logits to predicted class indices
    preds = np.argmax(logits, axis=-1)

    # Compute binary classification metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [17]:
# Create Trainer and train on augmented dataset
tuned_dropout = float(best_hyperparameters.get("dropout", 0.1))  # Get tuned dropout

def model_init_textattack():  # Define model init for Trainer
    return BertForSequenceClassification.from_pretrained(  # Load model from saved checkpoint
        model_checkpoint_path,                      # Set checkpoint path
        num_labels=2,                               # Set number of labels
        hidden_dropout_prob=tuned_dropout,          # Set tuned dropout
        attention_probs_dropout_prob=tuned_dropout, # Set tuned dropout
    )

training_args = TrainingArguments(  # Define training arguments
    output_dir=("results_textattack_augmented"),  # Set output directory
    eval_strategy="epoch",          # Evaluate every epoch
    save_strategy="epoch",          # Save every epoch
    load_best_model_at_end=True,    # Load best model
    metric_for_best_model="f1",     # Select best by F1
    greater_is_better=True,         # Higher F1 is better
    learning_rate=float(best_hyperparameters["learning_rate"]),          # Set learning rate
    num_train_epochs=int(best_hyperparameters["num_train_epochs"]),      # Set epochs
    per_device_train_batch_size=int(best_hyperparameters["per_device_train_batch_size"]),          # Set train batch size
    per_device_eval_batch_size=max(32, int(best_hyperparameters["per_device_train_batch_size"])),  # Set eval batch size
    weight_decay=float(best_hyperparameters.get("weight_decay", 0.01)),  # Set weight decay
    logging_steps=200,  # Set logging frequency
    seed=SEED,          # Set seed
    report_to=[],       # Disable external logging
)

trainer = Trainer(  # Create Trainer
    model_init=model_init_textattack,     # Pass model init
    args=training_args,                   # Pass args
    train_dataset=train_dataset_aug,      # Pass train dataset
    eval_dataset=val_dataset,             # Pass eval dataset
    tokenizer=tokenizer,                  # Pass tokenizer
    compute_metrics=compute_metrics,      # Enable metric reporting
)


/tmp/ipykernel_2569/3056292196.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(  # Create Trainer


In [ ]:
# Only run training when needed
train_output = trainer.train()  # Start model fine-tuning on augmented dataset

# Save the newly trained BERT model and tokenizer
FINAL_DIR = Path("../models/saved_model_augmented")   # Define final save directory for model and tokenizer
FINAL_DIR.mkdir(parents=True, exist_ok=True)          # Create directory if it doesn't exist

trainer.save_model(str(FINAL_DIR))                    # Save fine-tuned model weights and config
tokenizer.save_pretrained(str(FINAL_DIR))             # Save tokenizer files

print("Saved augmented model and tokenizer to:", FINAL_DIR)

# Evaluate on the held-out test set
test_results = trainer.evaluate(test_dataset)
print("Test Results:", test_results)

In [ ]:
# Run model predictions on test dataset
po = trainer.predict(test_dataset)          # Get model outputs (logits + labels)
logits = po.predictions                     # Extract raw logits from prediction output
labels = po.label_ids                       # Extract true labels from prediction output
preds = np.argmax(logits, axis=-1)          # Convert logits to predicted class indices

# Prepare results dictionary for JSON export
results = {"eval_samples": int(len(labels))}  # Record number of evaluated samples

# Create classification report and merge into results dict
report = classification_report(labels, preds,
                               target_names=["Ham", "Phishing"],
                               output_dict=True)  # Produce dict-formatted classification report
results.update(report)  # Merge classification metrics into results dict

# Save metrics and report to JSON
with open("test_results_augmented.json", "w") as f:  # Open JSON output file for writing
    json.dump(results, f, indent=2)  # Write results dict to file with indentation

# Configure confusion matrix display parameters
label_order = [0, 1]              # Ensure fixed label ordering: 0=Ham, 1=Phishing
disp_names = ["Ham", "Phishing"]  # Display names for confusion matrix axes

# Raw confusion matrix (integer counts)
cm = confusion_matrix(labels, preds, labels=label_order)                       # Compute raw confusion matrix counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=disp_names)  # Create display object
disp.plot(cmap="Blues", values_format="d")      # Plot matrix using integer formatting
plt.title("Confusion Matrix on Augmented Set")  # Add plot title
plt.tight_layout()                              # Adjust layout to avoid clipping
plt.savefig("confusion_matrix_raw_augmented.png", bbox_inches="tight", dpi=300)  # Save raw confusion matrix image
plt.show()            # Display plot inline
plt.close()           # Close figure to free memory

# Normalized confusion matrix (proportions by true label)
cm_norm = confusion_matrix(labels, preds, labels=label_order, normalize="true")     # Compute normalized confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=disp_names)  # Create display object for normalized matrix
disp.plot(cmap="Blues")  # Plot normalized matrix
plt.title("Normalized Confusion Matrix (Proportions)")  # Add title for normalized view
plt.tight_layout()                                      # Adjust layout
plt.savefig("confusion_matrix_normalized_augmented.png", bbox_inches="tight", dpi=300)  # Save normalized confusion matrix image
plt.show()            # Display normalized plot inline
plt.close()           # Close figure to free memory
